# 🧪 Held-Out Function Evaluation
## 测试模型对从未见过的函数的泛化能力

**目的**：训练时完全排除 10 个函数，测试时评估模型在这些 unseen 函数上的表现，回答"模型是记住了函数映射，还是真正学会了 function calling 推理"。

**环境**：Google Colab L4 GPU

## 1. 环境安装

In [ ]:
%%capture
!pip install -U transformers>=4.44.0
!pip install -U peft>=0.12.0
!pip install -U bitsandbytes>=0.43.0
!pip install -U accelerate>=0.33.0
!pip install -U datasets>=2.20.0
!pip install -U scipy
!pip install -U protobuf
print("All packages installed!")

## 2. 导入库 & 检查 GPU

In [ ]:
import torch
import json
import re
import random
import numpy as np
from datetime import datetime
from collections import defaultdict, Counter
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    bf16_support = torch.cuda.get_device_capability(0)[0] >= 8
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"BF16 support: {bf16_support}")
else:
    print("No GPU!")

## 3. 实验配置

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATASET_NAME = "glaiveai/glaive-function-calling-v2"
MAX_TRAIN_SAMPLES = 10000
MAX_TEST_SAMPLES = 500
MAX_SEQ_LENGTH = 512

LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_STEPS = 50

OUTPUT_DIR = "./fc_holdout_experiment"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8 if torch.cuda.is_available() else False
USE_FP16 = not USE_BF16

# 要 hold out 的函数（覆盖不同难度类型）
HOLDOUT_FUNCTIONS = {
    "calculate_bmi",          # 简单数值（v3 中 97% 准确率）
    "get_stock_price",        # 简单字符串（v3 中 100%）
    "search_recipe",          # 列表参数（v3 中 21%）
    "generate_qr_code",       # 混合参数（v3 中 61%）
    "calculate_distance",     # 嵌套参数（v3 中 42%）
    "send_email",             # 多字符串参数
    "get_movie_details",      # 字符串查询（v3 中 86%）
    "translate_text",         # 多字符串参数
    "set_alarm",              # 混合类型
    "calculate_area",         # 数值计算（v3 中 58%）
}

print("Experiment: Held-Out Function Evaluation")
print(f"Model: {MODEL_ID}")
print(f"Holdout functions ({len(HOLDOUT_FUNCTIONS)}): {HOLDOUT_FUNCTIONS}")
print(f"Precision: {'bf16' if USE_BF16 else 'fp16'}")

## 4. 加载与处理数据集

In [ ]:
print("Loading Glaive Function Calling v2 dataset...")
raw_dataset = load_dataset(DATASET_NAME, split="train")
print(f"Total samples: {len(raw_dataset)}")

In [ ]:
def parse_glaive_sample(sample):
    system_prompt = sample.get("system", "")
    chat = sample.get("chat", "")
    if "<functioncall>" not in chat:
        return None
    parts = chat.split("ASSISTANT:")
    if len(parts) < 2:
        return None
    user_part = parts[0]
    user_match = re.search(r'USER:\s*(.*)', user_part, re.DOTALL)
    if not user_match:
        return None
    user_query = user_match.group(1).strip()
    fc_text = None
    for part in parts[1:]:
        if "<functioncall>" in part:
            fc_text = part
            break
    if fc_text is None:
        return None
    fc_match = re.search(r'<functioncall>\s*(.*?)\s*<\|endoftext\|>', fc_text, re.DOTALL)
    if not fc_match:
        fc_match = re.search(r'<functioncall>\s*(.*)', fc_text, re.DOTALL)
    if not fc_match:
        return None
    raw_fc = fc_match.group(1).strip()
    fc_json = None
    try:
        fc_json = json.loads(raw_fc)
    except json.JSONDecodeError:
        pass
    if fc_json is None:
        try:
            fixed = re.sub(r"'(\{.*?\})'", r'\1', raw_fc)
            fc_json = json.loads(fixed)
        except:
            pass
    if fc_json is None:
        try:
            fixed = raw_fc.replace("'", '"')
            fc_json = json.loads(fixed)
        except:
            return None
    if not isinstance(fc_json, dict) or "name" not in fc_json:
        return None
    args = fc_json.get("arguments", {})
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except:
            args = {}
    if args is None:
        args = {}
    clean_fc = json.dumps({"name": fc_json["name"], "arguments": args}, ensure_ascii=False)
    return {
        "system": system_prompt.strip(),
        "user": user_query,
        "function_call": clean_fc,
        "function_name": fc_json["name"]
    }

print("Parsing dataset...")
parsed_data = []
failed = 0
for sample in raw_dataset:
    result = parse_glaive_sample(sample)
    if result:
        parsed_data.append(result)
    else:
        failed += 1
print(f"Successfully parsed: {len(parsed_data)}")
print(f"Skipped: {failed}")

## 5. Held-Out 数据分割

关键步骤：将数据按函数名分为两组，确保训练集中**完全不包含** holdout 函数。

In [ ]:
# 按函数名分割
train_pool = [s for s in parsed_data if s["function_name"] not in HOLDOUT_FUNCTIONS]
unseen_pool = [s for s in parsed_data if s["function_name"] in HOLDOUT_FUNCTIONS]

random.shuffle(train_pool)
random.shuffle(unseen_pool)

# 训练集：只用 seen 函数
train_parsed = train_pool[:MAX_TRAIN_SAMPLES]

# 测试集 - seen 函数：从训练池剩余部分取（不与训练重叠）
test_seen = train_pool[MAX_TRAIN_SAMPLES:MAX_TRAIN_SAMPLES + MAX_TEST_SAMPLES]

# 测试集 - unseen 函数：从 holdout 池取
test_unseen = unseen_pool[:MAX_TEST_SAMPLES]

print("=" * 55)
print("DATA SPLIT SUMMARY")
print("=" * 55)
print(f"  Train (seen functions only):  {len(train_parsed)}")
print(f"  Test - SEEN functions:        {len(test_seen)}")
print(f"  Test - UNSEEN functions:      {len(test_unseen)}")

# 验证：确认训练集中没有 holdout 函数
train_func_names = set(s["function_name"] for s in train_parsed)
leaked = train_func_names & HOLDOUT_FUNCTIONS
if leaked:
    print(f"\n  ERROR: Data leak detected! {leaked}")
else:
    print(f"\n  ✅ No data leakage: holdout functions completely excluded from training")

# 统计
print(f"\n  Train covers {len(train_func_names)} unique functions")
print(f"  Top 10 train functions:")
train_counts = Counter(s["function_name"] for s in train_parsed)
for fname, count in train_counts.most_common(10):
    print(f"    {fname}: {count}")

print(f"\n  Unseen test function distribution:")
unseen_counts = Counter(s["function_name"] for s in test_unseen)
for fname, count in sorted(unseen_counts.items(), key=lambda x: -x[1]):
    print(f"    {fname}: {count}")

## 6. 加载 Tokenizer & 模型

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"
print(f"Tokenizer loaded. EOS: {repr(tokenizer.eos_token)}")

In [ ]:
# 格式化训练数据（含 EOS token）
def format_training_text(sample):
    text = f"""### System:
{sample['system']}

When you need to call a function, respond ONLY with a JSON object in this exact format:
{{"name": "function_name", "arguments": {{"arg1": "value1"}}}}
Do not include any other text before or after the JSON.

### User:
{sample['user']}

### Assistant:
{sample['function_call']}{tokenizer.eos_token}"""
    return {"text": text}

formatted_data = [format_training_text(s) for s in train_parsed]
train_dataset = Dataset.from_list(formatted_data)
print(f"Train dataset: {len(train_dataset)} samples (seen functions only)")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)
model = prepare_model_for_kbit_training(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded! Parameters: {total_params / 1e9:.2f}B")

## 7. 配置 LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")

## 8. 训练

In [ ]:
def tokenize_dataset(dataset, tokenizer, max_length):
    tokenized = []
    for sample in dataset:
        encoded = tokenizer(sample["text"], truncation=True, max_length=max_length, padding=False)
        encoded["labels"] = encoded["input_ids"].copy()
        tokenized.append(encoded)
    return Dataset.from_list(tokenized)

print(f"Tokenizing (max_length={MAX_SEQ_LENGTH})...")
train_dataset_tok = tokenize_dataset(train_dataset, tokenizer, MAX_SEQ_LENGTH)
lengths = [len(x["input_ids"]) for x in train_dataset_tok]
print(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")

In [ ]:
def custom_collator(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, attention_mask, labels = [], [], []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append([1] * len(x["input_ids"]) + [0] * pad_len)
        labels.append(x["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=USE_FP16,
    bf16=USE_BF16,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tok,
    data_collator=custom_collator,
)
print("Ready to train!")

In [ ]:
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Time: {train_result.metrics['train_runtime']:.0f}s")
print(f"  Final loss: {train_result.metrics['train_loss']:.4f}")

trainer.save_model(OUTPUT_DIR)
print(f"  Saved to: {OUTPUT_DIR}")

## 9. 评估：Seen vs Unseen 函数

核心实验：对比模型在训练时见过的函数和完全没见过的函数上的表现。

In [ ]:
# 切换到推理模式
model.gradient_checkpointing_disable()
model.config.use_cache = True
print("Switched to inference mode.")

# 生成函数
def generate_function_call(model, tokenizer, system_prompt, user_query, max_new_tokens=256):
    prompt = f"""### System:
{system_prompt}

When you need to call a function, respond ONLY with a JSON object in this exact format:
{{"name": "function_name", "arguments": {{"arg1": "value1"}}}}
Do not include any other text before or after the JSON.

### User:
{user_query}

### Assistant:
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    eos_ids = [tokenizer.eos_token_id]
    for special_token in ["<|im_end|>", "<|endoftext|>"]:
        token_id = tokenizer.convert_tokens_to_ids(special_token)
        if token_id != tokenizer.unk_token_id and token_id not in eos_ids:
            eos_ids.append(token_id)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=eos_ids,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# JSON 提取（只取第一个完整 JSON）
def extract_json_from_text(text):
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    depth = 0
    start = None
    for i, ch in enumerate(text):
        if ch == '{':
            if depth == 0: start = i
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0 and start is not None:
                try:
                    return json.loads(text[start:i+1])
                except json.JSONDecodeError:
                    start = None
    return None

# 评估函数
def evaluate_function_call(predicted_json, ground_truth_json):
    result = {"json_valid": predicted_json is not None, "name_correct": False,
              "args_name_correct": False, "args_value_correct": False, "exact_match": False}
    if predicted_json is None:
        return result
    gt = ground_truth_json
    result["name_correct"] = (predicted_json.get("name", "") == gt.get("name", ""))
    pred_args = predicted_json.get("arguments", {})
    gt_args = gt.get("arguments", {})
    if isinstance(pred_args, str):
        try: pred_args = json.loads(pred_args)
        except: pred_args = {}
    if isinstance(gt_args, str):
        try: gt_args = json.loads(gt_args)
        except: gt_args = {}
    pred_keys = set(pred_args.keys()) if isinstance(pred_args, dict) else set()
    gt_keys = set(gt_args.keys()) if isinstance(gt_args, dict) else set()
    result["args_name_correct"] = (pred_keys == gt_keys)
    if result["args_name_correct"] and isinstance(pred_args, dict) and isinstance(gt_args, dict):
        all_match = True
        for key in gt_keys:
            if str(pred_args.get(key, "")).strip().lower() != str(gt_args.get(key, "")).strip().lower():
                all_match = False
                break
        result["args_value_correct"] = all_match
    result["exact_match"] = (result["name_correct"] and result["args_value_correct"])
    return result

print("Evaluation functions defined.")

In [ ]:
# ============ 评估 SEEN 函数 ============
print("=" * 60)
print("EVALUATING ON SEEN FUNCTIONS (in-distribution)")
print("=" * 60)

seen_results = []
for i, sample in enumerate(test_seen):
    pred_text = generate_function_call(model, tokenizer, sample["system"], sample["user"])
    pred_json = extract_json_from_text(pred_text)
    gt_json = json.loads(sample["function_call"])
    eval_result = evaluate_function_call(pred_json, gt_json)
    eval_result["predicted_text"] = pred_text
    eval_result["ground_truth"] = sample["function_call"]
    eval_result["function_name"] = sample["function_name"]
    seen_results.append(eval_result)
    if (i + 1) % 50 == 0:
        acc = sum(r["exact_match"] for r in seen_results) / len(seen_results)
        print(f"  [{i+1}/{len(test_seen)}] Exact match: {acc:.1%}")

seen_n = len(seen_results)
seen_metrics = {
    "JSON Valid Rate": sum(r["json_valid"] for r in seen_results) / seen_n,
    "Function Name Acc": sum(r["name_correct"] for r in seen_results) / seen_n,
    "Arg Names Acc": sum(r["args_name_correct"] for r in seen_results) / seen_n,
    "Arg Values Acc": sum(r["args_value_correct"] for r in seen_results) / seen_n,
    "Exact Match Rate": sum(r["exact_match"] for r in seen_results) / seen_n,
}
print(f"\nSEEN Exact Match: {seen_metrics['Exact Match Rate']:.1%}")

In [ ]:
# ============ 评估 UNSEEN 函数 ============
print("=" * 60)
print("EVALUATING ON UNSEEN FUNCTIONS (out-of-distribution)")
print("=" * 60)

unseen_results = []
for i, sample in enumerate(test_unseen):
    pred_text = generate_function_call(model, tokenizer, sample["system"], sample["user"])
    pred_json = extract_json_from_text(pred_text)
    gt_json = json.loads(sample["function_call"])
    eval_result = evaluate_function_call(pred_json, gt_json)
    eval_result["predicted_text"] = pred_text
    eval_result["ground_truth"] = sample["function_call"]
    eval_result["function_name"] = sample["function_name"]
    unseen_results.append(eval_result)
    if (i + 1) % 50 == 0:
        acc = sum(r["exact_match"] for r in unseen_results) / len(unseen_results)
        print(f"  [{i+1}/{len(test_unseen)}] Exact match: {acc:.1%}")

unseen_n = len(unseen_results)
unseen_metrics = {
    "JSON Valid Rate": sum(r["json_valid"] for r in unseen_results) / unseen_n,
    "Function Name Acc": sum(r["name_correct"] for r in unseen_results) / unseen_n,
    "Arg Names Acc": sum(r["args_name_correct"] for r in unseen_results) / unseen_n,
    "Arg Values Acc": sum(r["args_value_correct"] for r in unseen_results) / unseen_n,
    "Exact Match Rate": sum(r["exact_match"] for r in unseen_results) / unseen_n,
}
print(f"\nUNSEEN Exact Match: {unseen_metrics['Exact Match Rate']:.1%}")

## 10. 汇总对比

In [ ]:
print("=" * 65)
print("GENERALIZATION ANALYSIS: Seen vs Unseen Functions")
print("=" * 65)
print(f"  {'Metric':<20} {'Seen':>12} {'Unseen':>12} {'Gap':>12}")
print("-" * 65)
for metric in seen_metrics:
    s = seen_metrics[metric]
    u = unseen_metrics[metric]
    gap = s - u
    print(f"  {metric:<20} {s:>11.1%} {u:>11.1%} {gap:>11.1%}")
print("=" * 65)

# 按 unseen 函数逐个统计
print("\nPER-FUNCTION ACCURACY (Unseen Functions)")
print("-" * 55)
func_stats = defaultdict(lambda: {"total": 0, "correct": 0})
for r in unseen_results:
    func_stats[r["function_name"]]["total"] += 1
    if r["exact_match"]:
        func_stats[r["function_name"]]["correct"] += 1

for fname, stats in sorted(func_stats.items(), key=lambda x: -x[1]["total"]):
    acc = stats["correct"] / stats["total"]
    bar = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
    print(f"  {fname:<30} {bar} {stats['correct']}/{stats['total']} ({acc:.0%})")

# 失败样例分析
print("\nFAILURE EXAMPLES (Unseen Functions)")
print("-" * 55)
count = 0
for r in unseen_results:
    if not r["exact_match"] and count < 5:
        print(f"  Func: {r['function_name']}")
        print(f"  GT:   {r['ground_truth'][:150]}")
        print(f"  Pred: {r['predicted_text'][:150]}")
        print(f"  JSON: {r['json_valid']} | Name: {r['name_correct']} | Args: {r['args_name_correct']}")
        print()
        count += 1

In [ ]:
# 保存结果
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

results_file = f"{OUTPUT_DIR}/holdout_results_{TIMESTAMP}.json"
with open(results_file, "w") as f:
    json.dump({
        "experiment": "held_out_function_evaluation",
        "model": MODEL_ID,
        "holdout_functions": sorted(list(HOLDOUT_FUNCTIONS)),
        "train_samples": len(train_parsed),
        "test_seen_samples": len(test_seen),
        "test_unseen_samples": len(test_unseen),
        "seen_metrics": seen_metrics,
        "unseen_metrics": unseen_metrics,
        "generalization_gap": {k: seen_metrics[k] - unseen_metrics[k] for k in seen_metrics},
    }, f, indent=2)
print(f"Results saved to: {results_file}")

# 保存到 Google Drive
from google.colab import drive
drive.mount('/content/drive')
import shutil

save_dir = "/content/drive/MyDrive/FC_Experiments/holdout_experiment"
os.makedirs(save_dir, exist_ok=True)
shutil.copytree(OUTPUT_DIR, save_dir, dirs_exist_ok=True)
print(f"Saved to Google Drive: {save_dir}")

## 🎯 实验完成！

**如何解读结果**：
- Seen 和 Unseen 的 Exact Match 差距 < 10%：说明模型学到了通用的 function calling 能力，泛化良好
- 差距 10-20%：泛化能力一般，但仍有实用价值
- 差距 > 20%：模型主要在记忆训练集中的函数映射

**论文用途**：这个实验直接回应"数据泄露/过拟合"的质疑，是论文 Section 5 最有说服力的补充实验。